In [19]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import xgboost as xgb

from pathlib import Path

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    KFold,
    cross_validate
)

from sklearn.dummy import DummyRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    max_error
)

# Reproducibility
RANDOM_SEED = 1
np.random.seed(RANDOM_SEED)

# Environment check
print("Python:", sys.version)
print("Python executable:", sys.executable)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("XGBoost:", xgb.__version__)
print("Random seed:", RANDOM_SEED)

Python: 3.10.21 (main, Sep  1 2026, 14:14:55) [MSC v.1944 64 bit (AMD64)]
Python executable: c:\Users\jinwo\Downloads\CHE1147-course\.venv\Scripts\python.exe
NumPy: 1.26.4
Pandas: 2.3.3
scikit-learn: 1.7.2
XGBoost: 3.2.0
Random seed: 1


In [20]:
# Locate the project directory
PROJECT_DIR = Path.cwd()

if not (PROJECT_DIR / "Extraction").is_dir():
    PROJECT_DIR = PROJECT_DIR.parent

# Load the extracted dataset
DATA_DIR = PROJECT_DIR / "Extraction"
INPUT_FILE = DATA_DIR / "extracted_data.csv"

df = pd.read_csv(
    INPUT_FILE,
    keep_default_na=False,
    na_values=[""]
)

print("Dataset path:", INPUT_FILE)
print("Dataset shape:", df.shape)

display(df.head())

Dataset path: c:\Users\jinwo\Downloads\Nuclear-Half-Life-Project\Extraction\extracted_data.csv
Dataset shape: (3371, 20)


,nuclide,element,Z,A,N,dataset_name,ground_energy_keV,spin_parity_raw,half_life_raw,half_life_value,half_life_unit,half_life_seconds,half_life_status,Q_beta_keV,S_n_keV,S_p_keV,Q_alpha_keV,ground_state_ambiguous,ground_state_candidate_count,source_file
0,1H,H,1,1,0,ADOPTED LEVELS,0.0,1/2+,STABLE,NaN,NaN,NaN,stable,NaN,NaN,NaN,NaN,False,1,ensdf.001
1,2H,H,1,2,1,ADOPTED LEVELS,0.0,1+,STABLE,NaN,NaN,NaN,stable,NaN,2224.566,2224.566,1.0,False,1,ensdf.002
2,3H,H,1,3,2,ADOPTED LEVELS,0.0,1/2+,12.32 Y,12.32,Y,388789632.0,finite,18.5906,6257.233,NaN,NaN,False,1,ensdf.003
3,3HE,HE,2,3,1,ADOPTED LEVELS,0.0,1/2+,STABLE,NaN,NaN,NaN,stable,NaN,NaN,5493.477,NaN,False,1,ensdf.003
4,4H,H,1,4,3,ADOPTED LEVELS,0.0,2-,NaN,NaN,NaN,NaN,missing,NaN,NaN,NaN,NaN,False,1,ensdf.004


In [21]:
# Keep isotopes with finite half-lives
reg_df = df[df["half_life_status"] == "finite"].copy()

# Transform half-life into log10(seconds)
reg_df["log_half_life"] = np.log10(
    reg_df["half_life_seconds"]
)

print("Original samples:", len(df))
print("Removed samples:", len(df) - len(reg_df))
print("Finite half-life samples:", len(reg_df))
print("Invalid target values:",
      (~np.isfinite(reg_df["log_half_life"])).sum())

display(
    reg_df[["nuclide", "half_life_seconds", "log_half_life"]].head()
)

Original samples: 3371
Removed samples: 575
Finite half-life samples: 2796
Invalid target values: 0


,nuclide,half_life_seconds,log_half_life
2,3H,3.887896e+08,8.589715
11,6HE,8.067000e-01,-0.093288
17,7BE,4.598208e+06,6.662589
19,8HE,1.191000e-01,-0.924088
20,8LI,8.399000e-01,-0.075772


In [22]:
# Calculate neutron-to-proton ratio
reg_df["N/Z"] = reg_df["N"] / reg_df["Z"]

# Define input features and target
FEATURES = [
    "A",
    "N/Z",
    "Q_beta_keV",
    "Q_alpha_keV",
    "S_n_keV",
    "S_p_keV"
]

TARGET = "log_half_life"

X = reg_df[FEATURES].copy()
y = reg_df[TARGET].copy()

# Check dataset dimensions
print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

# Check missing values
missing_summary = pd.DataFrame({
    "Available": X.notna().sum(),
    "Missing": X.isna().sum(),
    "Coverage (%)": X.notna().mean() * 100
})

display(missing_summary.round(2))

print("Complete samples:", X.dropna().shape[0])

Feature matrix shape: (2796, 6)
Target shape: (2796,)


,Available,Missing,Coverage (%)
A,2796,0,100.00
N/Z,2796,0,100.00
Q_beta_keV,1970,826,70.46
Q_alpha_keV,2181,615,78.00
S_n_keV,2172,624,77.68
S_p_keV,2166,630,77.47


Complete samples: 1671


In [23]:
# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_SEED
)

# Define 5-fold cross-validation
cv = KFold(
    n_splits=5,
    shuffle=False
)

print("Random seed:", RANDOM_SEED)
print("Training samples:", len(X_train))
print("Test samples:", len(X_test))
print("CV folds:", cv.get_n_splits())

Random seed: 1
Training samples: 2236
Test samples: 560
CV folds: 5


In [24]:
# Define regression evaluation metrics
def evaluate_regression(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MSE": mean_squared_error(y_true, y_pred),
        "Max Error": max_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred)
    }


# Evaluate training and test performance
def evaluate_model(model):
    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    results = pd.DataFrame({
        "Train": evaluate_regression(y_train, train_pred),
        "Test": evaluate_regression(y_test, test_pred)
    }).T

    return results


print("Evaluation functions are ready.")

Evaluation functions are ready.


In [25]:
dummy_results = []

for strategy in ["mean", "median"]:

    model = DummyRegressor(strategy=strategy)

    # Cross-validation
    cv_results = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring="neg_mean_absolute_error",
        return_train_score=True
    )

    # Train the model
    model.fit(X_train, y_train)

    # Evaluate train and test performance
    results = evaluate_model(model)

    dummy_results.append({
        "Model": f"Dummy ({strategy})",
        "Train MAE": results.loc["Train", "MAE"],
        "CV MAE": -cv_results["test_score"].mean(),
        "CV Std": cv_results["test_score"].std(),
        "Test MAE": results.loc["Test", "MAE"],
        "Test RMSE": results.loc["Test", "RMSE"],
        "Test MSE": results.loc["Test", "MSE"],
        "Test Max Error": results.loc["Test", "Max Error"],
        "Test R2": results.loc["Test", "R2"]
    })

dummy_df = pd.DataFrame(dummy_results)

display(dummy_df.round(6))

,Model,Train MAE,CV MAE,CV Std,Test MAE,Test RMSE,Test MSE,Test Max Error,Test R2
0,Dummy (mean),2.425752,2.426242,0.089396,2.358409,3.172637,10.065623,12.480676,-0.000028
1,Dummy (median),2.425363,2.425784,0.088808,2.356491,3.172811,10.066727,12.426691,-0.000138


In [26]:
# Decision Tree baseline
# Evaluate the default tree using 5-fold CV on the training set.

dt_baseline = DecisionTreeRegressor(
    random_state=RANDOM_SEED
)

dt_baseline_cv = cross_validate(
    dt_baseline,
    X_train,
    y_train,
    cv=5,
    scoring="neg_mean_absolute_error"
)

dt_baseline_cv_mae = -dt_baseline_cv["test_score"].mean()

print("Decision Tree baseline CV MAE:")
print(dt_baseline_cv_mae)

Decision Tree baseline CV MAE:
1.2461979840473945


In [27]:
# Decision Tree hyperparameter optimization

dt_param_grid = {
    "max_depth": [None, 5, 10, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 5]
}

dt_grid = GridSearchCV(
    estimator=DecisionTreeRegressor(
        random_state=RANDOM_SEED
    ),
    param_grid=dt_param_grid,
    cv=5,
    scoring="neg_mean_absolute_error",
    n_jobs=2,
    verbose=1
)

dt_grid.fit(X_train, y_train)

print("Best parameters:")
print(dt_grid.best_params_)

print("\nBest CV MAE:")
print(-dt_grid.best_score_)

Fitting 5 folds for each of 36 candidates, totalling 180 fits
Best parameters:
{'max_depth': 20, 'min_samples_leaf': 2, 'min_samples_split': 5}

Best CV MAE:
1.2094463038510461


In [28]:
# Fit the baseline Decision Tree
dt_baseline.fit(X_train, y_train)

# Select the optimized Decision Tree
dt_best = dt_grid.best_estimator_

# Evaluate both models
dt_baseline_results = evaluate_model(dt_baseline)
dt_best_results = evaluate_model(dt_best)

# Combine evaluation results
dt_results = pd.concat(
    {
        "Decision Tree - Base": dt_baseline_results,
        "Decision Tree - Opt": dt_best_results
    },
    names=["Model", "Dataset"]
)

display(dt_results.round(6))

MAE      RMSE       MSE  Max Error  \
Model                Dataset                                            
Decision Tree - Base Train    0.016800  0.207684  0.043133   6.400415   
                     Test     1.113828  1.861560  3.465405   8.878266   
Decision Tree - Opt  Train    0.369195  0.681819  0.464877   8.215253   
                     Test     1.061456  1.702946  2.900025   9.175544   

                                    R2  
Model                Dataset            
Decision Tree - Base Train    0.995824  
                     Test     0.655709  
Decision Tree - Opt  Train    0.954994  
                     Test     0.711880

In [29]:
# Random Forest baseline

rf_baseline = RandomForestRegressor(
    random_state=RANDOM_SEED,
    n_jobs=1
)

rf_baseline_cv = cross_validate(
    rf_baseline,
    X_train,
    y_train,
    cv=5,
    scoring="neg_mean_absolute_error",
    n_jobs=2
)

rf_baseline_cv_mae = -rf_baseline_cv["test_score"].mean()

print("Random Forest baseline CV MAE:")
print(rf_baseline_cv_mae)

Random Forest baseline CV MAE:
0.9821899478120777


In [30]:
# Random Forest hyperparameter optimization

rf_param_grid = {
    "n_estimators": [100, 300],
    "max_depth": [None, 10, 20],
    "min_samples_leaf": [1, 2]
}

rf_grid = GridSearchCV(
    estimator=RandomForestRegressor(
        random_state=RANDOM_SEED,
        n_jobs=1
    ),
    param_grid=rf_param_grid,
    cv=5,
    scoring="neg_mean_absolute_error",
    n_jobs=2,
    verbose=1
)

rf_grid.fit(X_train, y_train)

print("Best parameters:")
print(rf_grid.best_params_)

print("\nBest CV MAE:")
print(-rf_grid.best_score_)

Fitting 5 folds for each of 12 candidates, totalling 60 fits
Best parameters:
{'max_depth': None, 'min_samples_leaf': 1, 'n_estimators': 300}

Best CV MAE:
0.9815709763225726


In [31]:
# Fit the baseline Random Forest
rf_baseline.fit(X_train, y_train)

# Select the optimized Random Forest
rf_best = rf_grid.best_estimator_

# Evaluate both models
rf_baseline_results = evaluate_model(rf_baseline)
rf_best_results = evaluate_model(rf_best)

# Combine evaluation results
rf_results = pd.concat(
    {
        "Random Forest - Base": rf_baseline_results,
        "Random Forest - Opt": rf_best_results
    },
    names=["Model", "Dataset"]
)

display(rf_results.round(6))

MAE      RMSE       MSE  Max Error  \
Model                Dataset                                            
Random Forest - Base Train    0.359737  0.562988  0.316956   5.818751   
                     Test     0.893089  1.412638  1.995545   8.643464   
Random Forest - Opt  Train    0.353594  0.554398  0.307357   6.421325   
                     Test     0.887499  1.408120  1.982802   8.528848   

                                    R2  
Model                Dataset            
Random Forest - Base Train    0.969315  
                     Test     0.801741  
Random Forest - Opt  Train    0.970244  
                     Test     0.803007

In [32]:
# XGBoost baseline

xgb_baseline = XGBRegressor(
    objective="reg:squarederror",
    random_state=RANDOM_SEED,
    tree_method="hist",
    n_jobs=1
)

xgb_baseline_cv = cross_validate(
    xgb_baseline,
    X_train,
    y_train,
    cv=5,
    scoring="neg_mean_absolute_error",
    n_jobs=2
)

xgb_baseline_cv_mae = -xgb_baseline_cv["test_score"].mean()

print("XGBoost baseline CV MAE:")
print(xgb_baseline_cv_mae)

XGBoost baseline CV MAE:
1.005610989057188


In [33]:
# XGBoost hyperparameter optimization

xgb_param_grid = {
    "n_estimators": [100, 300],
    "max_depth": [3, 5],
    "learning_rate": [0.01, 0.1],
    "subsample": [0.8, 1.0]
}

xgb_grid = GridSearchCV(
    estimator=XGBRegressor(
        objective="reg:squarederror",
        random_state=RANDOM_SEED,
        tree_method="hist",
        n_jobs=1
    ),
    param_grid=xgb_param_grid,
    cv=5,
    scoring="neg_mean_absolute_error",
    n_jobs=2,
    verbose=1,
    error_score="raise"
)

xgb_grid.fit(X_train, y_train)

print("Best parameters:")
print(xgb_grid.best_params_)

print("\nBest CV MAE:")
print(-xgb_grid.best_score_)

Fitting 5 folds for each of 16 candidates, totalling 80 fits
Best parameters:
{'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 300, 'subsample': 0.8}

Best CV MAE:
0.9352608199921567


In [34]:
# Fit the baseline XGBoost
xgb_baseline.fit(X_train, y_train)

# Select the optimized XGBoost
xgb_best = xgb_grid.best_estimator_

# Evaluate both models
xgb_baseline_results = evaluate_model(xgb_baseline)
xgb_best_results = evaluate_model(xgb_best)

# Combine evaluation results
xgb_results = pd.concat(
    {
        "XGBoost - Base": xgb_baseline_results,
        "XGBoost - Opt": xgb_best_results
    },
    names=["Model", "Dataset"]
)

display(xgb_results.round(6))

MAE      RMSE       MSE  Max Error        R2
Model          Dataset                                                   
XGBoost - Base Train    0.187102  0.273215  0.074647   1.791023  0.992773
               Test     0.878134  1.355420  1.837165   8.785091  0.817476
XGBoost - Opt  Train    0.329574  0.462181  0.213611   2.574190  0.979320
               Test     0.830708  1.285362  1.652157   7.638107  0.835857

In [35]:
# Final model comparison

model_list = [
    ("Decision Tree - Base", dt_baseline, dt_baseline_cv_mae),
    ("Decision Tree - Opt", dt_best, -dt_grid.best_score_),
    ("Random Forest - Base", rf_baseline, rf_baseline_cv_mae),
    ("Random Forest - Opt", rf_best, -rf_grid.best_score_),
    ("XGBoost - Base", xgb_baseline, xgb_baseline_cv_mae),
    ("XGBoost - Opt", xgb_best, -xgb_grid.best_score_)
]

final_results = []

for name, model, cv_mae in model_list:

    results = evaluate_model(model)

    final_results.append({
        "Model": name,
        "Train MAE": results.loc["Train", "MAE"],
        "CV MAE": cv_mae,
        "Test MAE": results.loc["Test", "MAE"],
        "Test RMSE": results.loc["Test", "RMSE"],
        "Test MSE": results.loc["Test", "MSE"],
        "Test Max Error": results.loc["Test", "Max Error"],
        "Test R2": results.loc["Test", "R2"],
        "Train-Test Gap": (
            results.loc["Test", "MAE"]
            - results.loc["Train", "MAE"]
        )
    })

final_df = pd.DataFrame(final_results)

final_df = final_df.sort_values(
    "CV MAE"
).reset_index(drop=True)

display(final_df.round(6))

,Model,Train MAE,CV MAE,Test MAE,Test RMSE,Test MSE,Test Max Error,Test R2,Train-Test Gap
0,XGBoost - Opt,0.329574,0.935261,0.830708,1.285362,1.652157,7.638107,0.835857,0.501134
1,Random Forest - Opt,0.353594,0.981571,0.887499,1.408120,1.982802,8.528848,0.803007,0.533904
2,Random Forest - Base,0.359737,0.982190,0.893089,1.412638,1.995545,8.643464,0.801741,0.533352
3,XGBoost - Base,0.187102,1.005611,0.878134,1.355420,1.837165,8.785091,0.817476,0.691032
4,Decision Tree - Opt,0.369195,1.209446,1.061456,1.702946,2.900025,9.175544,0.711880,0.692261
5,Decision Tree - Base,0.016800,1.246198,1.113828,1.861560,3.465405,8.878266,0.655709,1.097027
